# Train a CLI Agent with GRPO using Unsloth and NeMo Gym

This notebook trains **Nemotron-Nano-9B-v2** to translate natural language into LangGraph CLI commands using:
- **Unsloth** for efficient training
- **NeMo Gym** resource server for verifiable rewards
- **GRPO** (Group Relative Policy Optimization) for RL training

## What You'll Learn

1. Start the NeMo Gym resource server for CLI verification
2. Load Nemotron-Nano-9B-v2
3. Create a reward function that calls the NeMo Gym `/verify` endpoint
4. Train with GRPO
5. Save the trained model locally

## Step 1: Start the NeMo Gym Resource Server

Before training, start the LangGraph CLI resource server in a separate terminal:

```bash
cd nemo_gym_resources/langgraph_cli
uvicorn app:app --host 0.0.0.0 --port 8001
```

You should see:
```
INFO:     Uvicorn running on http://0.0.0.0:8000
INFO:     Application startup complete.
```

The server provides a `/verify` endpoint that evaluates model outputs against expected CLI commands.

In [ ]:
import requests

# NeMo Gym resource server configuration
NEMO_GYM_HOST = "127.0.0.1"
NEMO_GYM_PORT = 8001
VERIFY_ENDPOINT = f"http://{NEMO_GYM_HOST}:{NEMO_GYM_PORT}/verify"

# Verify server is running
try:
    response = requests.get(f"http://{NEMO_GYM_HOST}:{NEMO_GYM_PORT}/health", timeout=5)
    if response.status_code == 200:
        print(f"NeMo Gym server is running at {NEMO_GYM_HOST}:{NEMO_GYM_PORT}")
        print(f"Verify endpoint: {VERIFY_ENDPOINT}")
    else:
        print(f"Server responded with status {response.status_code}")
except requests.exceptions.ConnectionError:
    print("ERROR: NeMo Gym server is not running!")
    print("Please start it with: uvicorn app:app --host 0.0.0.0 --port 8001")
    print("(from the nemo_gym_resources/langgraph_cli directory)")

## Step 2: Verify Environment

Dependencies are pre-installed during project build (see `postBuild.bash`). Run this cell to verify your environment is ready for GRPO training.

The cell prints your architecture and compute capability so you can confirm the build targeted your GPU:
- **A100** → compute cap `(8, 0)`
- **H100** → compute cap `(9, 0)`
- DGX Spark GB10 → compute cap `(12, 1)`

> Bolded options represent recommended hardware for running this notebook!

In [ ]:
# Dependencies are pre-installed during project build (see postBuild.bash).
# This cell verifies the environment is ready.
import sys, platform
import torch

print(f"Python:       {sys.version.split()[0]}")
print(f"Architecture: {platform.machine()}")
print(f"PyTorch:      {torch.__version__}")
print(f"CUDA avail:   {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU:          {torch.cuda.get_device_name(0)}")
    print(f"Compute cap:  {torch.cuda.get_device_capability(0)}")
    print(f"Arch list:    {torch.cuda.get_arch_list()}")

try:
    import unsloth, transformers, trl, bitsandbytes
    print(f"Unsloth:      {unsloth.__version__}")
    print(f"Transformers: {transformers.__version__}")
    print(f"TRL:          {trl.__version__}")
    print(f"bitsandbytes: {bitsandbytes.__version__}")
    print("\n✅ Environment ready for GRPO training")
except ImportError as e:
    print(f"\n❌ Import failed: {e}")

In [ ]:
import os
os.environ["TORCHDYNAMO_DISABLE"] = "1"  # Disable torch.compile for stability

import json
from typing import List, Dict, Any, Optional

import torch
import torch._dynamo
torch._dynamo.config.suppress_errors = True

import numpy as np
from datasets import Dataset
from unsloth import FastLanguageModel
from trl import GRPOConfig, GRPOTrainer

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## Step 3: Load Nemotron-Nano-9B with Unsloth + Monkey Patch

Nemotron-Nano-9B-v2 is a hybrid Transformer-Mamba2 model (~9B parameters).

**Important:** Nemotron uses custom modeling code (`modeling_nemotron_h.py`) that doesn't respect Unsloth's `UNSLOTH_RETURN_HIDDEN_STATES` environment variable. We apply a monkey-patch to fix this.

We use BF16 (no 4-bit quantization) because Mamba2 CUDA kernels don't support quantization.

In [ ]:
# Model configuration
MODEL_NAME = "nvidia/NVIDIA-Nemotron-Nano-9B-v2"
MODEL_REVISION = "6533e8de2c68e4536bf7c411d7a3ce5734111476"
MAX_SEQ_LENGTH = 1024
LORA_RANK = 16

print(f"Loading model: {MODEL_NAME}")

# NOTE: Nemotron-Nano-9B-v2 is a hybrid Transformer-Mamba2 model.
# The Mamba2 CUDA kernels are NOT compatible with bitsandbytes 4-bit quantization.
# Use BF16. Check available memory before loading; shared GPUs may not fit this run.

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    revision=MODEL_REVISION,
    max_seq_length=MAX_SEQ_LENGTH,
    trust_remote_code=True,
    load_in_8bit=False,
    load_in_4bit=False,  # Disabled: Mamba2 kernels incompatible with 4-bit
    dtype=torch.bfloat16,
)

print(f"Model loaded successfully!")

In [ ]:
# Apply the Nemotron monkey-patch for Unsloth GRPO compatibility
from nemotron_unsloth_patch import patch_nemotron_for_unsloth_grpo

patch_nemotron_for_unsloth_grpo(model)

# Apply LoRA adapters for parameter-efficient training
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_RANK,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=LORA_RANK * 2,
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

# Ensure pad token is set
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

print(f"LoRA adapters applied (rank={LORA_RANK})")

## Step 4: Load Training Dataset

Load synthetic CLI data. Each record contains:
- `input`: Natural language request
- `output`: Expected structured CLI tool call (JSON)

In [ ]:
# System prompt for CLI tool-calling.

import os
import sys

LAB_DIR = os.getcwd()  # Jupyter starts the kernel in the notebook's directory
if not os.path.isdir(os.path.join(LAB_DIR, "bash_agent")):
    raise RuntimeError(
        f"Expected to find bash_agent/ in {LAB_DIR}. "
        "Run this notebook from code/4-agent-customization/."
    )
if LAB_DIR not in sys.path:
    sys.path.insert(0, LAB_DIR)

from bash_agent.prompts import JSON_SYSTEM_PROMPT as SYSTEM_PROMPT

print(SYSTEM_PROMPT)


In [ ]:
from training_support import prepare_examples, check_split

def load_training_data(filepath: str) -> Dataset:
    with open(filepath) as f:
        records = prepare_examples(json.loads(line) for line in f if line.strip())
    return Dataset.from_list([{
        "prompt": [{"role": "system", "content": SYSTEM_PROMPT},
                   {"role": "user", "content": row["input"]}],
        "answer": row["output"],
        "user_input": row["input"],
    } for row in records])

from pathlib import Path
# Default: reviewed workshop data. After QA, select data/langgraph_cli/generated instead.
DATA_DIR = Path("data/langgraph_cli")
TRAIN_FILE = DATA_DIR / "train.jsonl"
VAL_FILE = DATA_DIR / "val.jsonl"
train_dataset = load_training_data(TRAIN_FILE)
val_dataset = load_training_data(VAL_FILE)
check_split([{"input": x["user_input"]} for x in train_dataset],
            [{"input": x["user_input"]} for x in val_dataset])
print(f"Loaded {len(train_dataset)} training and {len(val_dataset)} held-out examples")


## Step 5: Create Reward Function Using NeMo Gym Server

The reward function calls the NeMo Gym resource server's `/verify` endpoint to evaluate model outputs.

What happens in NeMo Gym during training:
- Send model response to the verification server
- Server computes reward based on correctness
- Return reward for GRPO training

### 📋 Exercise

Implement the reward function by making a call to the `/verify` endpoint. 

This is the bridge between GRPO and verifiable rewards: each model completion gets sent to the NeMo Gym server, which returns a reward score (gate-then-grade: invalid JSON or a wrong command scores −1, otherwise the score scales with flag accuracy). Implement `resp` by posting a request to `verify_endpoint` with `json` set to `verify_request` and the `timeout` set to 30s.

<details>
<summary>💡 NEED SOME HELP?</summary>

---

```python
resp = requests.post(verify_endpoint, json=verify_request, timeout=30)
```
</details>

In [ ]:
def create_nemo_gym_reward_function(verify_endpoint: str):
    def reward_fn(completions, answer=None, prompts=None, **kwargs):
        # TRL repeats each row's answer alongside its generated completions.
        # This also works on held-out rows: no lookup by user wording is needed.
        if answer is None or len(answer) != len(completions):
            raise ValueError("Each completion needs its dataset answer column.")
        rewards = []
        for i, (completion, expected) in enumerate(zip(completions, answer, strict=True)):
            text = completion[0]["content"] if completion else ""
            user_query = next((m["content"] for m in (prompts[i] if prompts else [])
                               if m["role"] == "user"), "CLI evaluation")
            verify_request = {
                "task_id": f"completion-{i}",
                "task_input": {"input": user_query, "output": expected},
                "model_response": text,
            }
            # TODO: Send the request to the NeMo Gym resource server.
            resp = ...
            # Infrastructure failures stop the run; they are not model rewards.
            resp.raise_for_status()
            reward = float(resp.json()["reward"])
            if not np.isfinite(reward) or not -1 <= reward <= 1:
                raise ValueError("Verifier returned an invalid reward.")
            rewards.append(reward)
        return np.array(rewards)
    return reward_fn

reward_fn = create_nemo_gym_reward_function(VERIFY_ENDPOINT)
print(f"Reward function ready: {VERIFY_ENDPOINT}")


### Test the Reward Function

Verify that the reward function correctly communicates with the NeMo Gym server.

In [ ]:
# Test the verify endpoint directly
test_request = {
    "task_id": "test-1",
    "task_input": {
        "input": "Build a Docker image and tag it as myapp:latest",
        "output": {"command": "build", "tag": "myapp:latest"}
    },
    "model_response": '{"command": "build", "tag": "myapp:latest"}'
}

try:
    resp = requests.post(VERIFY_ENDPOINT, json=test_request, timeout=10)
    print(f"Status: {resp.status_code}")
    print(f"Response: {json.dumps(resp.json(), indent=2)}")
except Exception as e:
    print(f"Error: {e}")
    print("Make sure the NeMo Gym server is running!")

## Step 6: Configure and Run GRPO Training

**GRPO (Group Relative Policy Optimization)** samples multiple completions per prompt
and reinforces outputs that perform better relative to the group.

The NeMo Gym server provides consistent, verifiable rewards for each completion.

**Note:** With our monkey-patch applied, Unsloth's efficient GRPO implementation now works with Nemotron.

Before configuring the trainer, measure the formatted prompts in both splits and reserve at least 256 completion tokens within `MAX_SEQ_LENGTH`. This check stops on overlong requests so training does not silently truncate inputs or change the held-out comparison.


In [ ]:
# Verify the monkey-patch is working
from nemotron_unsloth_patch import verify_patch
verify_patch(model, tokenizer)

In [ ]:
# Size prompts and completions before creating the training configuration.
from training_support import training_token_budget

MIN_COMPLETION_BUDGET = 256
MAX_PROMPT_LENGTH, MAX_COMPLETION_LENGTH = training_token_budget(
    tokenizer, train_dataset, val_dataset, MAX_SEQ_LENGTH,
    min_completion_length=MIN_COMPLETION_BUDGET,
)
print(f"Max prompt length: {MAX_PROMPT_LENGTH}")
print(f"Max completion length: {MAX_COMPLETION_LENGTH}")
print(f"Total sequence budget: {MAX_PROMPT_LENGTH + MAX_COMPLETION_LENGTH}")


### 📋 Exercise

Implement some key training configuration parameters. 

These three settings control the core training dynamics: `num_generations` is how many candidate outputs GRPO generates per prompt (more = richer comparison signal), `learning_rate` controls the step size for weight updates, and `max_steps` caps the total training iterations. Implement `training_args` with `num_generations=4`, `learning_rate=1e-5`, and `max_steps=50`.

<details>
<summary>💡 NEED SOME HELP?</summary>

---

```python
training_args = GRPOConfig(
    ... # Keep other args as-is
    num_generations=4,
    ...
    learning_rate=1e-5,
    ...
    max_steps=50,
    ...
)
```
</details>

In [ ]:
# TODO: Fill in the blanks for the GRPO Training Configuration
training_args = GRPOConfig(
    # Generation settings
    temperature=1.0,
    num_generations= ... ,  # Completions per prompt for GRPO
    max_prompt_length=MAX_PROMPT_LENGTH,
    max_completion_length=MAX_COMPLETION_LENGTH,
    
    # Training settings
    learning_rate= ... ,  # Rate at which the model learns
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    num_train_epochs=1,
    max_steps= ... ,  # Increase for better results, decrease for faster training
    warmup_ratio=0.1,
    weight_decay=0.01,
    
    # Optimizer
    optim="adamw_8bit",
    lr_scheduler_type="cosine",
    
    # GRPO settings
    epsilon_high=0.28,
    mask_truncated_completions=True,
    
    # Logging
    logging_steps=1,
    save_steps=50,
    output_dir="outputs/grpo_langgraph_cli",
    report_to="none",
)

print("GRPO configuration:")
print(f"  - Generations per prompt: {training_args.num_generations}")
print(f"  - Max steps: {training_args.max_steps}")
print(f"  - Learning rate: {training_args.learning_rate}")

### 📋 Exercise

Implement the `trainer` as a `GRPOTrainer` and wire up everything we've defined so far. 

The `GRPOTrainer` orchestrates the full training loop shown above: generate completions, score them via the reward function, and reinforce the best ones. Implement `trainer` with `model`, `processing_class` set to `tokenizer`, `reward_funcs` as a single-item list containing `reward_fn`, `args` set to `training_args`, and the `train_dataset`.

<details>
<summary>💡 NEED SOME HELP?</summary>

---

```python
trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=[reward_fn],
    args=training_args,
    train_dataset=train_dataset,
)
```
</details>

In [ ]:
# TODO: Create GRPO Trainer
trainer = GRPOTrainer(
    model= ... ,
    processing_class= ... ,
    reward_funcs= ... ,
    args= ... ,
    train_dataset= ... ,
)
print("GRPO Trainer ready!")
print(f"Training on {len(train_dataset)} samples")
print(f"Rewards computed via NeMo Gym server at {VERIFY_ENDPOINT}")

In [ ]:
# Freeze the held-out comparison before training this same local model.
from training_support import evaluate_cli_model
from pathlib import Path

EVAL_TOKENS = MIN_COMPLETION_BUDGET
with model.disable_adapter():
    baseline_eval = evaluate_cli_model(model, tokenizer, val_dataset, VERIFY_ENDPOINT, EVAL_TOKENS)
print(f"Base 9B held-out exact match: {baseline_eval['exact_match_rate']:.1%}")

print("Starting GRPO. Equal rewards within a group give no relative learning signal.")
trainer.train()


## Step 7: Test the Trained Model

Score the trained model on the same held-out rows, system prompt, greedy decoding and completion budget as the base model. Inspect both improvements and regressions; a completed training run does not guarantee better answers.


In [ ]:
# Same model family, rows, prompt, greedy decoding and reward function.
trained_eval = evaluate_cli_model(model, tokenizer, val_dataset, VERIFY_ENDPOINT, EVAL_TOKENS)
comparison = {"base_model": MODEL_NAME, "base_revision": MODEL_REVISION,
              "baseline": baseline_eval, "trained": trained_eval}
comparison_path = Path("outputs/grpo_langgraph_cli/held_out_comparison.json")
comparison_path.parent.mkdir(parents=True, exist_ok=True)
comparison_path.write_text(json.dumps(comparison, indent=2))
print(f"Held-out exact match: {baseline_eval['exact_match_rate']:.1%} → "
      f"{trained_eval['exact_match_rate']:.1%}")
print(f"Inspect successes and regressions in {comparison_path}")


## Step 8: Save the Trained Model Locally

Save the merged model (base + LoRA weights) locally for use with the CLI agent.

In [ ]:
# Save adapters first, then merge the live BF16 model and write fresh weights.
from peft import PeftModel
from model_export import save_merged_model

OUTPUT_DIR = "outputs/grpo_langgraph_cli/merged_model"
ADAPTER_DIR = "outputs/grpo_langgraph_cli/final_adapter"

# Keep the trained adapter if export is interrupted. After merging, `model`
# is a regular inference model; rerun the loading/LoRA cells to train again.
# This guard also lets you retry this cell after a disk or save error.
if isinstance(model, PeftModel):
    model.save_pretrained(ADAPTER_DIR, safe_serialization=True)
    tokenizer.save_pretrained(ADAPTER_DIR)
    model = model.merge_and_unload(safe_merge=True)

print(f"Saving merged model to: {OUTPUT_DIR}")
save_merged_model(model, tokenizer, OUTPUT_DIR)

print("Model saved successfully!")
print("\nThis matches bash_agent's default model_path (config.py), so 'python -m bash_agent.main_hf' will load it automatically.")


## Summary

This notebook demonstrated:

1. **NeMo Gym Integration** - Connected to the resource server's `/verify` endpoint for reward computation
2. **Unsloth + Nemotron Fix** - Applied `nemotron_unsloth_patch.py` to support `UNSLOTH_RETURN_HIDDEN_STATES`
3. **GRPO Training** - Reinforcement learning with verifiable rewards using Unsloth's efficient trainer
4. **Local Model Saving** - Saved merged model for deployment

**Key Fix:** Nemotron uses custom modeling code that doesn't respect Unsloth's `UNSLOTH_RETURN_HIDDEN_STATES` environment variable. The `nemotron_unsloth_patch.py` module applies a monkey-patch to make it compatible.

## Architecture

```
┌─────────────────┐      ┌──────────────────────┐
│  GRPO Trainer   │────▶│  NeMo Gym Server     │
│  (Unsloth)      │      │  /verify endpoint    │
└────────┬────────┘      └──────────┬───────────┘
         │                          │
         │  completions             │  rewards
         │                          │
         ▼                          ▼
┌─────────────────────────────────────────────┐
│           Training Loop                     │
│  1. Generate completions for prompts        │
│  2. Send to NeMo Gym for verification       │
│  3. Compute GRPO loss from rewards          │
│  4. Update model weights (LoRA only)        │
└─────────────────────────────────────────────┘
```

## Next Steps

- Run the customized model with human-in-the-loop execution: `python -m bash_agent.main_hf` (or continue to notebook 03)
- Generate more synthetic training data with NeMo Data Designer
- Increase training steps for better performance

## References

- [NeMo Gym Documentation](https://docs.nvidia.com/nemo/gym/latest/index.html)
- [TRL GRPO Documentation](https://huggingface.co/docs/trl/main/en/grpo_trainer)
- [GRPO Paper](https://arxiv.org/abs/2402.03300)